# Actividad — Boosting: AdaBoost y Gradient Boosting
**Aprendizaje Supervisado · Sesión 14 · Boosting: AdaBoost, Gradient Boosting y XGBoost**

**Qué haremos.** En la sesión anterior (Unidad 13) vimos ensambles **en paralelo** (bagging, Random Forest). Aquí vemos ensambles **en serie**: cada modelo nuevo corrige los errores del anterior. Probamos AdaBoost y Gradient Boosting sobre el mismo tipo de conjunto sintético y los comparamos contra el Random Forest de la actividad anterior.

Trabaja de forma **individual**. Donde veas `...` con un comentario `# ✏️`, completa esa línea; donde veas **✏️ Pregunta**, responde en la celda de markdown siguiente.

## Parte 0 — Los datos

Usamos el mismo tipo de conjunto sintético de clasificación binaria de la Unidad 13 (800 observaciones, 10 variables, solo 5 informativas), y entrenamos de una vez el Random Forest de referencia de esa sesión para poder comparar directamente contra boosting más adelante.

In [ ]:
import warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier,
                               AdaBoostClassifier, GradientBoostingClassifier)
warnings.filterwarnings("ignore")

X, y = make_classification(n_samples=800, n_features=10, n_informative=5, n_redundant=2,
                            n_clusters_per_class=2, class_sep=1.1, random_state=0)
cols = [f"x{i}" for i in range(10)]
X = pd.DataFrame(X, columns=cols)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
print("Entrenamiento:", Xtr.shape, " Prueba:", Xte.shape, " Proporción clase 1:", round(y.mean(), 3))

# Modelo de referencia de la Unidad 13 (Random Forest), para comparar contra boosting
rf_referencia = RandomForestClassifier(n_estimators=300, max_features="sqrt",
                                        random_state=0, n_jobs=-1).fit(Xtr, ytr)
print("Random Forest de referencia (Unidad 13) — exactitud de prueba:", round(rf_referencia.score(Xte, yte), 3))


## Parte 1 — AdaBoost  *(presentación A1–A6)*

Entrenamos `AdaBoostClassifier` con *stumps* (árboles de profundidad 1) y graficamos el error de entrenamiento y de prueba contra el número de aprendices.

In [ ]:
n_est_list = [1, 5, 10, 20, 40, 80, 150, 250]
ada_tr, ada_te = [], []
for ne in n_est_list:
    ada = ...     # ✏️ AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=ne, random_state=0).fit(Xtr, ytr)
    ada_tr.append(1 - ada.score(Xtr, ytr))
    ada_te.append(1 - ada.score(Xte, yte))

plt.figure(figsize=(5.5, 3))
plt.plot(n_est_list, ada_tr, "o--", label="entrenamiento")
plt.plot(n_est_list, ada_te, "o-", label="prueba")
plt.xlabel("número de aprendices"); plt.ylabel("error"); plt.legend(); plt.title("AdaBoost")
plt.grid(alpha=0.3); plt.show()
print(f"Error de prueba con 1 aprendiz: {ada_te[0]:.3f}  |  con 250: {ada_te[-1]:.3f}")


✏️ **Pregunta 1.1.** (a) ¿El error de entrenamiento llega a 0? ¿Qué nos dice eso sobre el riesgo de sobreajuste de AdaBoost con muchas rondas (A5 de la presentación)? (b) Compara el mejor error de prueba de AdaBoost con el del Random Forest de referencia (`rf_referencia`, Unidad 13): ¿cuál es menor?

**Respuesta:**

## Parte 2 — Gradient Boosting y su ajuste  *(presentación B1–B3)*

A diferencia de AdaBoost, Gradient Boosting tiene dos hiperparámetros que interactúan fuertemente: `learning_rate` y `n_estimators`. Usamos `GridSearchCV` para elegir una buena combinación.

In [ ]:
malla = {"learning_rate": [0.01, 0.05, 0.1, 0.3], "n_estimators": [50, 100, 200]}
busqueda = ...     # ✏️ GridSearchCV(GradientBoostingClassifier(max_depth=3, random_state=0), malla, cv=kf, n_jobs=-1).fit(Xtr, ytr)

print("Mejores hiperparámetros:", busqueda.best_params_)
print(f"Exactitud CV del mejor modelo: {busqueda.best_score_:.3f}")
gbm_final = busqueda.best_estimator_


✏️ **Pregunta 2.1.** ¿Qué combinación de `learning_rate` y `n_estimators` ganó? Dado lo que vimos en la presentación (B3) sobre el intercambio entre ambos, ¿tiene sentido esa combinación (learning rate chico con muchos árboles, o grande con pocos)?

**Respuesta:**

## Parte 3 — Comparación final y recomendación  *(presentación D1)*

Reunimos el Random Forest de referencia (Unidad 13) con AdaBoost y Gradient Boosting en una tabla con exactitud de prueba y tiempo de entrenamiento.

In [ ]:
modelos = {
    "Random Forest (Unidad 13)": rf_referencia,
    "AdaBoost": AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=150, random_state=0),
    "Gradient Boosting": gbm_final,
}
filas = []
for nombre, modelo in modelos.items():
    t0 = time.time()
    modelo.fit(Xtr, ytr)
    dt = time.time() - t0
    filas.append({"modelo": nombre, "exactitud_prueba": round(modelo.score(Xte, yte), 3), "tiempo_entrenamiento_s": round(dt, 3)})
tabla_final = pd.DataFrame(filas).sort_values("exactitud_prueba", ascending=False)
tabla_final.to_csv("entregable_unidad14.csv", index=False)
tabla_final


✏️ **Pregunta 3.1.** (6 a 8 líneas) Con la tabla anterior: (1) ¿qué modelo elegirías para este problema y por qué? (2) Si tuvieras datos con muchos valores atípicos o etiquetas ruidosas, ¿evitarías alguno de los tres? ¿Por qué (piensa en A5 de la presentación)? (3) ¿Qué harías distinto si el objetivo fuera maximizar exactitud sin importar el tiempo de entrenamiento (piensa en la sección C de la presentación, XGBoost)? (4) Según D1, ¿cuál de los tres es más fácil de paralelizar entre árboles, y por qué eso importa en la práctica?

**Respuesta:**